# Streamlit App Runthrough

## Setup and Configs

In [1]:
# setting the environment variables, the keys
import sys
import os

sys.path.insert(0, os.path.abspath('..'))

from config import set_environment
# for the keys - as explained early in chapter 2
set_environment()

## Document Loader

Imports

In [2]:
import logging
import os
import pathlib
import tempfile
from typing import Any

from langchain_community.document_loaders.epub import UnstructuredEPubLoader
from langchain_community.document_loaders.pdf import PyPDFLoader
from langchain_community.document_loaders.text import TextLoader
from langchain_community.document_loaders.word_document import UnstructuredWordDocumentLoader
from langchain_core.documents import Document

from streamlit.logger import get_logger

logging.basicConfig(encoding="utf-8", level=logging.INFO)
LOGGER = get_logger(__name__)

Class Loaders

In [3]:
class EPubReader(UnstructuredEPubLoader):
    def __init__(self, file_path: str | list[str], **unstructured_kwargs: Any):
        super().__init__(file_path, **unstructured_kwargs, mode="elements", strategy="fast")

class DocumentLoaderException(Exception):
    pass

class DocumentLoader:
    """Loads in a document with a supported extension."""

    supported_extensions = {
        ".pdf": PyPDFLoader,
        ".txt": TextLoader,
        ".epub": EPubReader,
        ".docx": UnstructuredWordDocumentLoader,
        ".doc": UnstructuredWordDocumentLoader,
    }

Load Doc Function

In [4]:
def load_document(temp_filepath: str) -> list[Document]:
    """Load a file and return it as a list of documents.

    Doesn't handle a lot of errors at the moment.
    """
    ext = pathlib.Path(temp_filepath).suffix

    loader = DocumentLoader.supported_extensions.get(ext)
    if not loader:
        raise DocumentLoaderException(
            f"Invalid extention type {ext}, cannot load this type of file"
        )
    
    loaded = loader(temp_filepath)
    docs = loaded.load()
    logging.info(docs)
    return docs

## LLMs

Imports

In [5]:
from langchain.embeddings import CacheBackedEmbeddings
from langchain.storage import LocalFileStore
from langchain_groq import ChatGroq
from langchain_openai import OpenAIEmbeddings

General LLM Setup

In [6]:
chat_model = ChatGroq(
    model="deepseek-r1-distill-llama-70b",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

store = LocalFileStore("./cache/")

underlying_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large",
)

# Avoid unnecessary costs by caching the embeddings.
EMBEDDINGS = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings, store, namespace=underlying_embeddings.model
)

/opt/anaconda3/envs/langchain_ai/lib/python3.10/site-packages/langchain/embeddings/cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


## Retrievers

Imports

In [9]:
import os
import tempfile
from typing import Any

from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

VECTOR_STORE = InMemoryVectorStore(embedding=EMBEDDINGS)

Retriever class

In [10]:
def split_documents(docs: list[Document]) -> list[Document]:
    """Split each document."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500, chunk_overlap=200,
    )
    return text_splitter.split_documents(docs)

In [11]:
class DocumentRetriever(BaseRetriever):
    """A retriever that contains the top k documents that contain the user query."""
    documents: list[Document] = []
    k: int = 5

    def model_post_init(self, ctx: Any) -> None:
        self.store_documents(self.documents)

    def store_documents(docs: list[Document]) -> None:
        """Add documents to the vector store"""
        splits = split_documents(docs)
        VECTOR_STORE.add_documents(splits)

    def add_uploaded_docs(self, uploaded_files):
        """Add uploaded documents."""
        docs = []
        with tempfile.TemporaryDirectory() as temp_dir:
            for file in uploaded_files:
                temp_filepath = os.path.join(temp_dir, file.name)
                # Write file content first
                with open(temp_filepath, "wb") as f:
                    f.write(file.getvalue())
                # Load document AFTER file is closed
                try:
                    docs.extend(load_document(temp_filepath))
                except Exception as e:
                    print(f"Failed to load {file.name}: {e}")
                    continue

    def _get_relevant_documents(
            self, query: str, *, run_manager: CallbackManagerForRetrieverRun
    ) -> list[Document]:
        """Sync implementations for retriever."""
        if len(self.documents) == 0:
            return []
        return VECTOR_STORE.similarity_search(query=query, k=self.k)

In [12]:
"""LangGraph for RAG.

CorpDocs with Citations: A Corporate Documentation Pipeline with RAG and Source Attribution
This single file contains the complete code to run a documentation generation system
using LangChain, LangGraph, and Gradio. In addition to generating and refining documentation,
this pipeline now retrieves and attaches citations to the final output.

Workflow Overview:
1. Generate an initial project documentation draft from a user's request.
2. Analyze the draft for compliance with corporate standards.
3. If issues are detected, prompt for LLM feedback.
4. Finalize the documentation (integrating any feedback).
5. Retrieve and append citations to the final document.
6. Output the fully revised document with inline source citations.

Note: More details on performance measurement and observability will be covered in Chapter 8.

"""

from typing import Annotated

from langchain_core.documents import Document
from langchain_core.messages import AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langgraph.checkpoint.memory import MemorySaver
from langgraph.constants import END
from langgraph.graph import START, StateGraph, add_messages
from typing_extensions import List, TypedDict

from chapter4.llms import chat_model
from chapter4.retriever import DocumentRetriever


system_prompt = (
    "You're a helpful AI assistant. Given a user question "
    "and some corporate document snippets, write documentation. "
    "If none of the documents is relevant to the question, "
    "mention that there's no relevant document, and then "
    "answer the question to the best of your knowledge."
    "\n\nHere are the corporate documents: "
    "{context}"
)

# Initialize the LangChain ChatGroq interface using the API key from environment variables.
retriever = DocumentRetriever()
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{question}"),
    ]
)


# Define state for application
class State(TypedDict):
    question: str
    context: List[Document]
    answer: str
    issues_report: str
    issues_detected: bool
    messages: Annotated[list, add_messages]


# Define application steps
def retrieve(state: State):
    retrieved_docs = retriever.invoke(state["messages"][-1].content)
    print(retrieved_docs)
    return {"context": retrieved_docs}


def generate(state: State):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = prompt.invoke(
        {"question": state["messages"][-1].content, "context": docs_content}
    )
    response = chat_model.invoke(messages)
    print(response.content)
    return {"answer": response.content}


def double_check(state: State):
    result = chat_model.invoke(
        [
            {
                "role": "user",
                "content": (
                    f"Review the following project documentation for compliance with our corporate standards. "
                    f"Return 'ISSUES FOUND' followed by any issues detected or 'NO ISSUES': {state['answer']}"
                ),
            }
        ]
    )

    # Extract actual response (after thinking block)
    content = result.content
    if "</think>" in content:
        actual_response = content.split("</think>", 1)[1].strip()
    else:
        actual_response = content.strip()

    if "ISSUES FOUND" in actual_response:
        print("issues detected")
        return {
            "issues_report": actual_response.split("ISSUES FOUND", 1)[1].strip(),
            "issues_detected": True,
        }
    print("no issues detected")
    return {"issues_report": "", "issues_detected": False}


# NODE: doc_finalizer
# Finalizes the documentation by incorporating feedback if available.
def doc_finalizer(state: State):
    """Finalize documentation by integrating human feedback."""
    if "issues_detected" in state and state["issues_detected"]:
        response = chat_model.invoke(
            [
                {
                    "role": "user",
                    "content": (
                        f"Revise the following documentation to address these feedback points: {state['issues_report']}\n"
                        f"Original Document: {state['answer']}\n"
                        f"Always return the full revised document, even if no changes are needed."
                    ),
                }
            ]
        )
        return {"messages": [AIMessage(response.content)]}
    return {"messages": [AIMessage(state["answer"])]}


# Compile application and test
graph_builder = StateGraph(State).add_sequence(
    [retrieve, generate, double_check, doc_finalizer]
)
graph_builder.add_edge(START, "retrieve")
graph_builder.add_edge("doc_finalizer", END)
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)
config = {"configurable": {"thread_id": "abc123"}}

In [13]:
"""Streamlit app

Run this as follows:
> PYTHONPATH=. streamlit run chapter4/streamlit_app.py
"""

import streamlit as st
from langchain_core.messages import HumanMessage

from chapter4.document_loader import DocumentLoader
from chapter4.rag import graph, config, retriever

# Set page configuration
st.set_page_config(page_title="Corporate Documentation Manager", layout="wide")

# Initialize session state for chat history and file management
if "chat_history" not in st.session_state:
    st.session_state.chat_history = []
if "uploaded_files" not in st.session_state:
    st.session_state.uploaded_files = []

# Display chat messages from history on app rerun
for message in st.session_state.chat_history:
    print(f"message: {message}")
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# Take all uploaded files
docs = retriever.add_uploaded_docs(st.session_state.uploaded_files)


def process_message(message):
    """Assistant response.

    Note: this ignores the previous messages

    There's some better way to stream this:
    for event in graph.stream(
            {"messages": HumanMessage(message)}, config=config
    ):
        print(event.key())
        if event.key == "doc_finalizer":
            for value in event.values():
                yield value["messages"][-1].content

    """
    response = graph.invoke({"messages": HumanMessage(message)}, config=config)
    return response["messages"][-1].content


# Project description using markdown
st.markdown(
    """
# 📄 CorpDocs with Citations

CorpDocs is your corporate documentation assistant. This tool generates detailed project documentation,
verifies compliance with corporate standards, and integrates human feedback when necessary. Finally,
it retrieves and attaches source citations to the final document.

**Workflow:**
1. **Generate Documentation:** Create an initial draft.
2. **Compliance Check:** Automatically review for adherence to corporate guidelines.
3. **Human Feedback:** If issues are detected, provide corrective feedback.
4. **Finalize Document:** Produce the revised document.
5. **Add Citations:** Append source citations to the document.

If you like this application, please give us a 5-star review on [Amazon](https://amzn.to/3X1xQbn)!
"""
)


# Create two columns for chat and file management
col1, col2 = st.columns([2, 1])

with col1:
    st.subheader("Chat Interface")

    # React to user input
    if user_message := st.chat_input("Enter your message:"):
        # Display user message in chat message container
        with st.chat_message("User"):
            st.markdown(user_message)
        # Add user message to chat history
        st.session_state.chat_history.append({"role": "User", "content": user_message})
        response = process_message(user_message)
        with st.chat_message("Assistant"):
            st.markdown(response)
        # Add response to chat history
        st.session_state.chat_history.append({"role": "Assistant", "content": response})

with col2:
    st.subheader("Document Management")

    # File uploader
    uploaded_files = st.file_uploader(
        "Upload Documents",
        type=list(DocumentLoader.supported_extensions),
        accept_multiple_files=True,
    )
    if uploaded_files:
        for file in uploaded_files:
            if file.name not in st.session_state.uploaded_files:
                st.session_state.uploaded_files.append(file)

2025-12-23 00:51:28.607 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-23 00:51:28.610 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-23 00:51:28.612 WARNING streamlit.runtime.state.session_state_proxy: Session state does not function when running a script without `streamlit run`
2025-12-23 00:51:28.614 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-23 00:51:28.615 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-23 00:51:28.615 WARNING streamlit.runtime.scriptrunner_utils.script_run_c